In [1]:
import numpy as np
import pandas as pd           

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.tree import DecisionTreeClassifier

In [5]:
df = pd.read_csv('train_titanic_data.csv')
df.head(2)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C


# ***Today's Plan: End-to-End Pipeline Architecture (Titanic Dataset)***

---

### Step-by-Step Pipeline Flow

* **Step 1: Missing Value Imputation (`ColumnTransformer 1`)**
  * Titanic dataset me `Age` aur `Embarked` columns ke andar missing values hain.
  * Sabse pehle pehla `ColumnTransformer` chalega jo `Age` (Numerical) aur `Embarked` (Categorical) columns ki missing values ko impute karega.

* **Step 2: One-Hot Encoding (`ColumnTransformer 2`)**
  * Step 1 se aane wala output second `ColumnTransformer` me as an input pass hoga.
  * Is step me do categorical columns (`Gender` / `Sex` aur `Embarked`) ke upar One-Hot Encoding perform hogi taaki textual categories binary numbers ($0, 1$) me badal jayein.

* **Step 3: Feature Scaling (`ColumnTransformer 3`)**
  * Step 2 ke output par scaling apply ki jayegi taaki saare columns ek similar numerical range me aa jayein aur koi ek feature model ke weights ko dominate na kare.

* **Step 4: Feature Selection**
  * Scaled features me se best **Top 5 columns** select kiye jayenge taaki model training fast ho aur irrelevant features filter ho jayein.

* **Step 5: Model Training (Decision Tree)**
  * Finally, Top 5 selected features ka output pass hoga **Decision Tree Classifier** ko, jahan model train ho kar predictions generate karega.

---

$$\text{Missing Imputation} \longrightarrow \text{One-Hot Encoding} \longrightarrow \text{Feature Scaling} \longrightarrow \text{Feature Selection (Top 5)} \longrightarrow \text{Decision Tree Training}$$

In [9]:
df.drop(columns=['PassengerId','Name','Ticket','Cabin'],inplace=True)

In [10]:
# step 1-> Train/Test/split 
X_train, X_test, y_train, y_test = train_test_split(df.drop(columns = ['Survived']), df['Survived'], test_size=0.2, random_state=42)

In [11]:
X_train.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
331,1,male,45.5,0,0,28.5000,S
733,2,male,23.0,0,0,13.0000,S
382,3,male,32.0,0,0,7.9250,S
704,3,male,26.0,1,0,7.8542,S
813,3,female,6.0,4,2,31.2750,S


In [12]:
y_train.head(5)

331    0
733    0
382    0
704    0
813    0
Name: Survived, dtype: int64

In [13]:
# impution tranformer 

trf1 = ColumnTransformer([
    ('imputer_age', SimpleImputer(),[2]),
    ('impute_embarked', SimpleImputer(strategy='most_frequent'),[6])
], remainder='passthrough')

In [ ]:
# one Hot Encoding
trf2 = ColumnTransformer([
    ('ohe_sex_embarked',OneHotEncoder(sparse_output=False, handle_unknown='ignore'),[1, 6])
], remainder='passthrough')

In [ ]:
# Scaling 
trf3 = ColumnTransformer([
    ('scale', MinMaxScaler(), slice(0, 8))
])

In [ ]:
# Feature selection
trf4 = SelectKBest(score_func=chi2, k=5)

In [16]:
# train the model 
trf5 = DecisionTreeClassifier()